# 📝 판다스 데이터 분석 기초 과제 LV2 정답 — 정제·변환·파생 조합 (강사용)

각 문제의 **모범답안 + 해설** 입니다.

> 참고: 이 레벨은 **중고차(used_cars)** 데이터가 기본이고, `to_datetime`·`map` 표기통일을 연습하는 **5·7번만** 그 개념에 필요한 컬럼(주문일시·회원등급)이 있는 **쇼핑몰(shop_orders)** 데이터를 씁니다. 문제마다 불러올 파일명을 확인하세요.

## 1. 조건에 맞는 상위 매물 뽑기 (필터 + 정렬)
**배경**: "무사고 가솔린 차 중 비싼 순 3대"를 뽑아 추천 목록을 만듭니다.

**요구사항**:
- `data/used_cars.csv` 를 `df` 로 불러오세요.
- `사고여부` 가 `'무사고'` 이고 `연료` 가 `'가솔린'` 인 매물만 거른 뒤, `가격만원` **내림차순**으로 정렬하고 **상위 3개**를 `top3` 에 담으세요.

**예시**
```
len(top3)                    →  3
top3.iloc[0]['가격만원']       →  4384   (가장 비싼 무사고 가솔린)
list(top3['가격만원'])         →  [4384, 4137, 4125]
```

<details><summary>힌트</summary>

```text
접근방법:
- 두 조건으로 먼저 거르고, 가격 내림차순으로 정렬한 뒤, head 로 위에서 3개만 남긴다.

세부구현:
1. (사고여부 무사고) 와 (연료 가솔린) 두 조건을 & 로 이어 거른다
2. sort_values 로 가격만원 내림차순 정렬한다
3. head(3) 으로 상위 3개를 top3 에 담는다
```

</details>

In [ ]:
import pandas as pd

df = pd.read_csv('../../day06_판다스_기초/data/used_cars.csv')
picked = df[(df['사고여부'] == '무사고') & (df['연료'] == '가솔린')]
top3 = picked.sort_values('가격만원', ascending=False).head(3)
print(list(top3['가격만원']))

In [ ]:
# [자가채점]
assert len(top3) == 3
assert top3.iloc[0]['가격만원'] == 4384
assert list(top3['가격만원']) == [4384, 4137, 4125]
print("✅ 문제1 통과!")

### 해설 — 문제 1
- **접근법**: 필터 → 정렬 → 상위 N 은 아주 흔한 3단 흐름입니다. `head(3)` 은 정렬된 결과의 위 3개를 가져와요.
- **흔한 실수**: 정렬 전에 `head(3)` 을 하면 "아무 3개"가 나옵니다. 반드시 **정렬 후** 잘라요.
- **대안**: `df.nlargest(3, '가격만원')` 은 필터 뒤 상위 3개를 한 번에 뽑는 지름길입니다.

## 2. 결측 행 버리고 자료형 바꾸기
**배경**: 색상이 빠진 매물은 이번 분석에서 제외하고, 주행거리는 숫자로 바꿔 계산할 준비를 합니다.

**요구사항**:
- `data/used_cars.csv` 를 `df` 로 불러오세요.
- `색상` 이 결측인 행을 버린 뒤(`dropna`), `주행거리` 의 콤마를 지우고 정수형으로 바꿔 `clean` 에 담으세요.

**예시**
```
len(clean)                 →  46   (50 − 색상 결측 4)
clean['주행거리'].iloc[0]    →  89000
```

<details><summary>힌트</summary>

```text
접근방법:
- 먼저 색상 결측 행을 dropna 로 버리고, 그 결과의 주행거리 열을 콤마 제거 후 정수형으로 바꾼다.

세부구현:
1. df.dropna(subset=['색상']) 뒤에 .copy() 를 붙여 복사본 clean 을 만든다 (원본 슬라이스에 바로 대입하면 경고가 날 수 있음)
2. clean['주행거리'] 를 .str.replace 로 콤마 제거
3. .astype(int) 로 정수형으로 바꿔 다시 clean['주행거리'] 에 넣는다
```

</details>

In [ ]:
import pandas as pd

df = pd.read_csv('../../day06_판다스_기초/data/used_cars.csv')
clean = df.dropna(subset=['색상']).copy()
clean['주행거리'] = clean['주행거리'].str.replace(',', '').astype(int)
print(len(clean), clean['주행거리'].iloc[0])

In [ ]:
# [자가채점]
assert len(clean) == 46, "색상 결측 4행을 버려 46행"
assert clean['주행거리'].iloc[0] == 89000
assert str(clean['주행거리'].dtype).startswith('int')
print("✅ 문제2 통과!")

### 해설 — 문제 2
- **접근법**: `dropna(subset=['색상'])` 은 색상이 빈 행만 버립니다. 이후 남은 행의 주행거리를 정수로 변환해요.
- **흔한 실수**: `dropna` 뒤에 `.copy()` 를 안 하면 원본 조각을 수정한다는 경고(SettingWithCopyWarning)가 뜰 수 있어요.
- **대안**: `subset` 없이 `dropna()` 만 하면 **어느 열이든** 결측이 있는 행을 모두 버려 데이터가 과하게 줄 수 있습니다.

## 3. 이름 공백 정리 후 요약 문구 만들기
**배경**: `모델` 값에 앞뒤 공백이 섞여 있어 그대로 쓰면 지저분합니다. 공백을 정리하고 보기 좋은 요약 문구를 만듭니다.

**요구사항**:
- `data/used_cars.csv` 를 `df` 로 불러오세요.
- `모델` 의 앞뒤 공백을 없앤 값을 `모델정제` 라는 새 열로 만드세요 (`.str.strip`).
- `모델정제` 와 `연식` 을 합쳐 `'모델정제(연식)'` 형태의 `요약` 열을 만드세요. (예: `소나타(2016)`)

**예시**
```
df.loc[0, '요약']  →  '소나타(2016)'
```

<details><summary>힌트</summary>

```text
접근방법:
- strip 으로 앞뒤 공백을 지운 열을 만들고, 문자열을 이어 붙여 요약 문구를 만든다. 숫자는 문자열로 바꿔 붙인다.

세부구현:
1. df['모델'].str.strip() 결과를 df['모델정제'] 에 넣는다
2. 연식을 문자열로 바꾼다 (astype(str))
3. '모델정제' + '(' + 연식문자열 + ')' 를 df['요약'] 에 넣는다
```

</details>

In [ ]:
import pandas as pd

df = pd.read_csv('../../day06_판다스_기초/data/used_cars.csv')
df['모델정제'] = df['모델'].str.strip()
df['요약'] = df['모델정제'] + '(' + df['연식'].astype(str) + ')'
print(df.loc[0, '요약'])

In [ ]:
# [자가채점]
assert df.loc[0, '요약'] == '소나타(2016)'
assert (df['모델정제'] == df['모델정제'].str.strip()).all(), "앞뒤 공백이 남아 있으면 안 돼요"
print("✅ 문제3 통과!")

### 해설 — 문제 3
- **접근법**: `.str.strip()` 은 앞뒤 공백만 지웁니다. 문자열끼리는 `+` 로 이어 붙일 수 있고, 숫자는 `astype(str)` 로 문자열로 바꿔야 붙어요.
- **흔한 실수**: 숫자 `연식` 을 문자열로 안 바꾸고 `+` 하면 타입이 안 맞아 에러가 납니다.
- **대안**: `df['연식'].astype(str)` 대신 f-string 을 apply 로 쓸 수도 있지만, 열 연산이 더 빠르고 간결합니다.

## 4. 이상치를 걸러 낸 통계
**배경**: `주행거리 999,999`, `가격만원 0` 같은 비현실적 값(이상치)이 평균을 왜곡합니다. 이들을 뺀 뒤 통계를 냅니다.

**요구사항**:
- `data/used_cars.csv` 를 `df` 로 불러오세요.
- `주행거리` 를 정수형으로 바꾼 뒤, **주행거리가 900000 미만이고 가격만원이 0보다 큰** 행만 남겨 `valid` 에 담으세요.
- `valid` 의 주행거리 평균과 가격만원 중앙값을 확인하세요.

**예시**
```
len(valid)                          →  48
round(valid['주행거리'].mean(), 1)   →  93062.5
valid['가격만원'].median()           →  2910.5
```

<details><summary>힌트</summary>

```text
접근방법:
- 주행거리를 정수로 바꾼 뒤, 두 임계 조건을 & 로 이어 이상치를 걸러 낸다. 그다음 mean 과 median 을 구한다.

세부구현:
1. 주행거리를 콤마 제거 후 정수형으로 바꾼다
2. (주행거리 < 900000) 와 (가격만원 > 0) 을 & 로 이어 valid 를 만든다
3. valid['주행거리'].mean() 과 valid['가격만원'].median() 을 확인한다
```

</details>

In [ ]:
import pandas as pd

df = pd.read_csv('../../day06_판다스_기초/data/used_cars.csv')
df['주행거리'] = df['주행거리'].str.replace(',', '').astype(int)
valid = df[(df['주행거리'] < 900000) & (df['가격만원'] > 0)]
print(len(valid), round(valid['주행거리'].mean(), 1), valid['가격만원'].median())

In [ ]:
# [자가채점]
assert len(valid) == 48, "주행 이상치 1건 + 가격 0 이상치 1건 제외 = 48"
assert round(valid['주행거리'].mean(), 1) == 93062.5
assert valid['가격만원'].median() == 2910.5
print("✅ 문제4 통과!")

### 해설 — 문제 4
- **접근법**: 이상치는 임계값으로 걸러 냅니다. 두 조건을 `&` 로 이어 한 번에 남겨요. `mean` 은 평균, `median` 은 중앙값입니다.
- **흔한 실수**: 이상치를 안 빼면 주행거리 평균이 `999,999` 때문에 확 커집니다. 그래서 평균보다 이상치에 둔감한 **중앙값**도 함께 봐요.
- **대안**: 임계 대신 상·하위 몇 %를 잘라 내는 방법(분위수)도 있지만, 여기서는 눈에 띄는 값을 직접 걸렀습니다.

## 5. 주문 일시에서 날짜 요소 뽑기 (to_datetime · dt)
**배경**: 쇼핑몰 주문일시가 `"2026-06-23 12:50"` 같은 **문자열**이라 날짜 계산이 안 됩니다. 날짜형으로 바꿔 시·일을 뽑습니다.

**요구사항**:
- `data/shop_orders.csv` 를 `df` 로 불러오세요.
- `주문일시` 를 `to_datetime` 으로 날짜형으로 바꿔 다시 `주문일시` 에 넣으세요.
- 시(hour)를 `시간` 열, 일(day)을 `일` 열로 뽑으세요. (`.dt.hour`, `.dt.day`)

**예시**
```
df['시간'].iloc[0]        →  12    (첫 주문 12:50)
(df['시간'] >= 18).sum()  →  28    (저녁 주문 수)
(df['일'] <= 10).sum()    →  24    (상순 주문 수)
```

<details><summary>힌트</summary>

```text
접근방법:
- 문자열 날짜를 to_datetime 으로 날짜형으로 바꾸면 .dt 접근자로 연·월·일·시를 꺼낼 수 있다.

세부구현:
1. pd.to_datetime(df['주문일시']) 결과를 df['주문일시'] 에 넣는다
2. df['주문일시'].dt.hour 를 df['시간'] 에 넣는다
3. df['주문일시'].dt.day 를 df['일'] 에 넣는다
```

</details>

In [ ]:
import pandas as pd

df = pd.read_csv('../../day06_판다스_기초/data/shop_orders.csv')
df['주문일시'] = pd.to_datetime(df['주문일시'])
df['시간'] = df['주문일시'].dt.hour
df['일'] = df['주문일시'].dt.day
print(df['시간'].iloc[0], (df['시간'] >= 18).sum(), (df['일'] <= 10).sum())

In [ ]:
# [자가채점]
assert df['시간'].iloc[0] == 12
assert (df['시간'] >= 18).sum() == 28, "18시 이상 저녁 주문 28건"
assert (df['일'] <= 10).sum() == 24, "1~10일 상순 주문 24건"
print("✅ 문제5 통과!")

### 해설 — 문제 5
- **접근법**: `to_datetime` 이 문자열을 날짜형으로 바꾸면 `.dt` 로 연/월/일/시/요일 등을 꺼낼 수 있습니다.
- **흔한 실수**: 날짜형으로 바꾸기 전에 `.dt` 를 쓰면 에러가 나요. `.dt` 는 datetime 열에서만 동작합니다.
- **대안**: 요일은 `.dt.dayofweek`(월=0), 월은 `.dt.month` 로 뽑습니다.

## 6. 사용자 함수로 차령(나이) 파생
**배경**: 연식만으로는 감이 안 오니, 올해(2026) 기준 **차령(연식으로부터 지난 햇수)** 을 만들어 봅니다.

**요구사항**:
- `data/used_cars.csv` 를 `df` 로 불러오세요.
- 연식 하나를 받아 `2026 - 연식` 을 돌려주는 함수 `car_age` 를 만드세요.
- 그 함수를 `연식` 열에 `apply` 해 `차령` 이라는 새 열을 만드세요.

**예시**
```
df.loc[0, '차령']   →  10   (2026 − 2016)
df['차령'].max()    →  11   (가장 오래된 차)
```

<details><summary>힌트</summary>

```text
접근방법:
- 연식 한 개를 받아 2026 에서 빼는 함수를 정의하고, 그 함수를 연식 열에 apply 해 새 열을 만든다.

세부구현:
1. car_age(year) 함수를 정의해 2026 - year 를 반환한다
2. df['연식'].apply(car_age) 결과를 df['차령'] 에 넣는다
3. df['차령'] 의 최댓값을 확인한다
```

</details>

In [ ]:
import pandas as pd

df = pd.read_csv('../../day06_판다스_기초/data/used_cars.csv')

def car_age(year):
    return 2026 - year

df['차령'] = df['연식'].apply(car_age)
print(df.loc[0, '차령'], df['차령'].max())

In [ ]:
# [자가채점]
assert df.loc[0, '차령'] == 10
assert df['차령'].max() == 11
print("✅ 문제6 통과!")

### 해설 — 문제 6
- **접근법**: `apply` 는 열의 각 값에 함수를 적용해 새 값을 만듭니다. 규칙이 조금만 복잡해도 함수로 빼면 읽기 좋아요.
- **흔한 실수**: `apply(car_age())` 처럼 괄호를 붙이면 함수를 **호출한 결과**를 넘겨 에러가 납니다. 함수 이름만 `apply(car_age)` 넘겨요.
- **대안**: 단순 뺄셈이라 `df['차령'] = 2026 - df['연식']` 처럼 열 연산으로도 됩니다. 규칙이 복잡할 때 `apply` 가 빛나요.

## 7. 회원등급 표기 통일하기 (map)
**배경**: `회원등급` 이 `gold / GOLD / Silver / SILVER / bronze / Bronze` 처럼 **대소문자가 뒤죽박죽**입니다. 표기를 대문자 3종으로 통일합니다.

**요구사항**:
- `data/shop_orders.csv` 를 `df` 로 불러오세요.
- 아래 매핑 딕셔너리로 `회원등급` 을 변환한 새 열 `등급정규화` 를 만드세요 (`map`).

```
grade_map = {
    'gold': 'GOLD', 'GOLD': 'GOLD',
    'silver': 'SILVER', 'SILVER': 'SILVER', 'Silver': 'SILVER',
    'bronze': 'BRONZE', 'BRONZE': 'BRONZE', 'Bronze': 'BRONZE',
}
```

**예시**
```
(df['등급정규화'] == 'SILVER').sum()  →  26
(df['등급정규화'] == 'BRONZE').sum()  →  24
(df['등급정규화'] == 'GOLD').sum()    →  20
```

<details><summary>힌트</summary>

```text
접근방법:
- map 은 딕셔너리의 키를 값으로 바꿔 준다. 원본에 나오는 모든 표기를 키로 넣어 두면 결측 없이 변환된다.

세부구현:
1. 지문의 grade_map 딕셔너리를 그대로 만든다
2. df['회원등급'].map(grade_map) 결과를 df['등급정규화'] 에 넣는다
3. 값별 개수를 확인한다
```

</details>

In [ ]:
import pandas as pd

df = pd.read_csv('../../day06_판다스_기초/data/shop_orders.csv')
grade_map = {
    'gold': 'GOLD', 'GOLD': 'GOLD',
    'silver': 'SILVER', 'SILVER': 'SILVER', 'Silver': 'SILVER',
    'bronze': 'BRONZE', 'BRONZE': 'BRONZE', 'Bronze': 'BRONZE',
}
df['등급정규화'] = df['회원등급'].map(grade_map)
print(df['등급정규화'].value_counts())

In [ ]:
# [자가채점]
assert (df['등급정규화'] == 'SILVER').sum() == 26
assert (df['등급정규화'] == 'BRONZE').sum() == 24
assert (df['등급정규화'] == 'GOLD').sum() == 20
assert df['등급정규화'].isna().sum() == 0, "매핑 안 된 값(결측)이 없어야 해요"
print("✅ 문제7 통과!")

### 해설 — 문제 7
- **접근법**: `map(딕셔너리)` 는 각 값을 딕셔너리의 대응 값으로 바꿉니다. 원본의 모든 표기를 키로 넣어야 빠짐없이 변환돼요.
- **흔한 실수**: 딕셔너리에 없는 표기가 있으면 그 자리는 `NaN` 이 됩니다. 그래서 `isna().sum()` 으로 빠진 게 없는지 확인해요.
- **대안**: 대소문자만 문제라면 `df['회원등급'].str.upper()` 한 줄로도 통일할 수 있습니다. `map` 은 임의의 값 매핑까지 되는 더 일반적인 도구예요.

## 8. 조건으로 등급 나누고 거르기 (np.where + 필터)
**배경**: 주행거리로 `고주행/저주행` 을 나눈 뒤, "고주행인데 유사고" 인 위험 매물을 추립니다.

**요구사항**:
- `data/used_cars.csv` 를 `df` 로 불러오세요.
- `주행거리` 를 정수형으로 바꾸고, **100000 이상이면 `'고주행'`, 아니면 `'저주행'`** 인 새 열 `주행등급` 을 `np.where` 로 만드세요.
- `주행등급` 이 `'고주행'` 이면서 `사고여부` 가 `'유사고'` 인 행만 골라 `high_risk` 에 담으세요.

**예시**
```
(df['주행등급'] == '고주행').sum()  →  21
len(high_risk)                    →  5
```

<details><summary>힌트</summary>

```text
접근방법:
- 주행거리를 정수로 바꾼 뒤, np.where(조건, 참일값, 거짓일값) 으로 등급 열을 만든다. 그다음 두 조건으로 위험 매물을 거른다.

세부구현:
1. numpy 를 np 로 불러오고 주행거리를 정수형으로 바꾼다
2. np.where(주행거리 >= 100000, '고주행', '저주행') 결과를 df['주행등급'] 에 넣는다
3. (주행등급 고주행) 와 (사고여부 유사고) 를 & 로 이어 high_risk 를 만든다
```

</details>

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('../../day06_판다스_기초/data/used_cars.csv')
df['주행거리'] = df['주행거리'].str.replace(',', '').astype(int)
df['주행등급'] = np.where(df['주행거리'] >= 100000, '고주행', '저주행')
high_risk = df[(df['주행등급'] == '고주행') & (df['사고여부'] == '유사고')]
print((df['주행등급'] == '고주행').sum(), len(high_risk))

In [ ]:
# [자가채점]
assert (df['주행등급'] == '고주행').sum() == 21
assert len(high_risk) == 5
print("✅ 문제8 통과!")

### 해설 — 문제 8
- **접근법**: `np.where(조건, A, B)` 는 조건이 참인 자리엔 A, 거짓인 자리엔 B 를 넣어 열을 한 번에 만듭니다. `if/else` 를 열 전체에 적용하는 셈이에요.
- **흔한 실수**: 앞 문제(999,999 이상치)를 안 지워도 이 문제는 `>= 100000` 조건이라 이상치가 '고주행'에 포함됩니다. 목적에 따라 이상치를 먼저 걸러야 할 수도 있어요.
- **대안**: `apply` 로도 되지만, 조건이 하나면 `np.where` 가 더 짧고 빠릅니다.

## 9. 중복 없는 모델 목록 (drop_duplicates + 정렬)
**배경**: 매물에는 같은 모델이 여러 번 나옵니다. **모델별로 하나씩만** 남긴 목록을 이름순으로 만듭니다.

**요구사항**:
- `data/used_cars.csv` 를 `df` 로 불러오세요.
- `모델` 의 앞뒤 공백을 없앤 `모델정제` 열을 만드세요.
- `모델정제` 기준 중복을 제거(첫 등장만 유지)한 뒤 `모델정제` 이름순(오름차순)으로 정렬해 `models` 에 담으세요.

**예시**
```
len(models)                    →  10   (서로 다른 모델 수)
models.iloc[0]['모델정제']       →  'K5'  (이름순 첫 모델)
```

<details><summary>힌트</summary>

```text
접근방법:
- 먼저 공백을 지운 모델정제 열을 만들고, 그 열 기준으로 중복을 제거한 뒤 이름순으로 정렬한다.

세부구현:
1. df['모델'].str.strip() 결과를 df['모델정제'] 에 넣는다
2. df.drop_duplicates(subset='모델정제') 로 중복을 제거한다
3. sort_values('모델정제') 로 이름순 정렬해 models 에 담는다
```

</details>

In [ ]:
import pandas as pd

df = pd.read_csv('../../day06_판다스_기초/data/used_cars.csv')
df['모델정제'] = df['모델'].str.strip()
models = df.drop_duplicates(subset='모델정제').sort_values('모델정제')
print(len(models), models.iloc[0]['모델정제'])

In [ ]:
# [자가채점]
assert len(models) == 10, "서로 다른 모델 10종"
assert models.iloc[0]['모델정제'] == 'K5'
print("✅ 문제9 통과!")

### 해설 — 문제 9
- **접근법**: `drop_duplicates(subset='모델정제')` 는 그 열 값이 처음 나온 행만 남깁니다. 공백을 먼저 지워야 `'소나타'` 와 `'소나타 '` 를 같은 것으로 봐요.
- **흔한 실수**: 공백 정리를 안 하면 앞뒤 공백 차이 때문에 같은 모델이 서로 다른 값으로 취급돼 중복이 덜 제거됩니다.
- **대안**: 마지막 등장을 남기려면 `keep='last'`, 완전히 같은 행 전체 기준이면 `subset` 을 생략합니다.

## 10. 조건에 맞는 매물 저장하기 (필터 → CSV)
**배경**: "무사고 디젤 차" 목록을 파일로 뽑아 팀에 공유합니다.

**요구사항**:
- `data/used_cars.csv` 를 `df` 로 불러오세요.
- `사고여부` 가 `'무사고'` 이고 `연료` 가 `'디젤'` 인 행만 골라 `result` 에 담으세요.
- `output/` 폴더가 없으면 만들고(`os.makedirs(..., exist_ok=True)`), `result` 를 `output/무사고_디젤.csv` 로 저장하세요 (`index=False`).

**예시**
```
len(result)  →  8
저장 경로: output/무사고_디젤.csv
```

<details><summary>힌트</summary>

```text
접근방법:
- 두 조건으로 거른 결과를 result 에 담고, output 폴더를 준비한 뒤 to_csv 로 저장한다.

세부구현:
1. (사고여부 무사고) 와 (연료 디젤) 을 & 로 이어 result 를 만든다
2. os.makedirs('output', exist_ok=True) 로 폴더를 준비한다
3. result.to_csv('output/무사고_디젤.csv', index=False) 로 저장한다
```

</details>

In [ ]:
import pandas as pd
import os

df = pd.read_csv('../../day06_판다스_기초/data/used_cars.csv')
result = df[(df['사고여부'] == '무사고') & (df['연료'] == '디젤')]
os.makedirs('output', exist_ok=True)
result.to_csv('output/무사고_디젤.csv', index=False)
print(len(result))

In [ ]:
# [자가채점]
assert len(result) == 8, "무사고 & 디젤 = 8대"
assert (result['사고여부'] == '무사고').all()
assert (result['연료'] == '디젤').all()
print("✅ 문제10 통과!")

### 해설 — 문제 10
- **접근법**: 거른 결과를 `to_csv(경로, index=False)` 로 저장합니다. `index=False` 는 판다스가 붙이는 행 번호를 파일에 안 쓰겠다는 뜻이에요.
- **흔한 실수**: `output/` 폴더가 없으면 저장이 실패합니다. `os.makedirs(경로, exist_ok=True)` 로 미리 만들어 둬요.
- **대안**: 다시 읽어 확인하려면 `pd.read_csv('output/무사고_디젤.csv')` 로 열어 행 수를 세어 봅니다.

## 11. 이상치를 잘라 담기 (clip · quantile)
**배경**: `주행거리 999,999` 처럼 비현실적으로 큰 값은 평균을 왜곡합니다. 행을 **지우는 대신**, 상한선으로 **눌러 담는(clip)** 방법도 있습니다. 어디를 상한으로 할지 감을 잡을 땐 `quantile`(분위수)이 유용해요.

**요구사항**:
- `data/used_cars.csv` 를 `df` 로 불러오고, `주행거리` 의 콤마를 지워 정수형으로 바꿔 `km` 에 담으세요.
- (참고) `km.quantile(0.95)` 로 상위 5% 경계가 얼마인지 출력해 보세요. (얼마나 큰 값이 섞여 있는지 감 잡기)
- 실무 기준 **30만(300000)km 를 상한**으로 정해, `km` 을 `clip(upper=300000)` 으로 눌러 담아 `capped` 에 넣으세요.

**예시**
```
km.quantile(0.95)       →  약 170400 (참고용, 정확히 안 맞아도 됨)
capped.max()            →  300000
round(capped.mean(), 1) →  96000.0
(km > 300000).sum()     →  1        (상한을 넘겨 눌린 매물 수)
```

<details><summary>힌트</summary>

```text
접근방법:
- 주행거리를 정수로 바꾼 뒤, quantile 로 상위 경계를 살펴보고, clip 의 upper 로 큰 값을 상한에 맞춰 누른다.

세부구현:
1. df['주행거리'] 를 .str.replace 로 콤마 제거 후 .astype(int) 로 바꿔 km 에 담는다
2. km.quantile(0.95) 를 출력해 참고한다
3. km.clip(upper=300000) 결과를 capped 에 담는다
```

</details>

In [ ]:
df = pd.read_csv('../../day06_판다스_기초/data/used_cars.csv')
km = df['주행거리'].str.replace(',', '').astype(int)
print("상위 95% 경계(참고):", km.quantile(0.95))
capped = km.clip(upper=300000)
print("clip 후 최댓값:", capped.max(), "| 평균:", round(capped.mean(), 1))
print("상한(30만)을 넘겨 눌린 매물:", (km > 300000).sum(), "대")

In [ ]:
# [자가채점]
assert capped.max() == 300000
assert round(capped.mean(), 1) == 96000.0
assert (km > 300000).sum() == 1, "30만km 초과 이상치는 1대"
print("✅ 문제11 통과!")